<a href="https://colab.research.google.com/github/noodlejacknetwork/Disaster-Prediction-2025/blob/main/disaster_events_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# DISASTER PREDICTION 2025 update as of March 12, 2026 @10:38am

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from scipy.stats import randint

#1 LOAD DATA
FILE_NAME = 'unclean_dataset.csv'

try:
    df = pd.read_csv(FILE_NAME)
    print(f"✅ Success: Loaded {FILE_NAME}")
except FileNotFoundError:
    print(f"❌ Error: {FILE_NAME} not found in the repository!")
    print("Please ensure the CSV file is uploaded to the same GitHub folder.")
    # stopper, if file missing
    raise

#2 PHASE 2 PREPROCESSIN
# Drop Id and processing dates
if 'event_id' in df.columns:
    df = df.drop(columns=['event_id'])

df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day_of_week'] = df['date'].dt.dayofweek
df = df.drop(columns=['date'])

# Log Transform for Population
df['affected_population'] = np.log1p(df['affected_population'])

# One-Hot Encoding
df = pd.get_dummies(df, columns=['disaster_type', 'location'], drop_first=True)

# PCA (Dimensionality Reduction)
pca_features = ['severity_level', 'affected_population', 'infrastructure_damage_index']
x_pca = StandardScaler().fit_transform(df[pca_features])
pca = PCA(n_components=2)
pcs = pca.fit_transform(x_pca)
df['total_impact_score'] = pcs[:, 0]
df['response_pattern_score'] = pcs[:, 1]

#3 PHASE 3 MODELINg
features = ['total_impact_score', 'response_pattern_score', 'estimated_economic_loss_usd', 'aid_provided', 'year', 'month', 'day_of_week']
# Adding the OHE columns
ohe_cols = [c for c in df.columns if 'disaster_type_' in c or 'location_' in c]
features.extend(ohe_cols)

X = df[features]
y = df['is_major_disaster']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# Training best model (Random Forest)
model = RandomForestClassifier(n_estimators=260, max_depth=20, min_samples_split=6, criterion='entropy', random_state=42)
model.fit(X_train, y_train)

#4. RESULTS with diagram
y_pred = model.predict(X_test)
print("\nModel Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# Diagram - Confusion Matrix
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix")
plt.show()

# Save model
joblib.dump(model, 'disaster_model_final.joblib')

❌ Error: unclean_dataset.csv not found in the repository!
Please ensure the CSV file is uploaded to the same GitHub folder.


FileNotFoundError: [Errno 2] No such file or directory: 'unclean_dataset.csv'